In [1]:
from keys import openai_key, replicate_key, groq_key
from working import JudgedDebate, get_client, Debater, Judge, CollabDebate

In [2]:
# n_debaters = 2

# debaters = [
#     Debater(
#         get_client("openai", openai_key),
#         f"debater {id}"
#     ) 
#     for id in range(n_debaters)
# ]

# judge = Judge(
#     get_client("openai", openai_key)
# )

# debate = JudgedDebate(debaters, judge)

In [3]:
import sys

sys.path.append("./mats_sae_training")
from sae_training.sparse_autoencoder import SparseAutoencoder


def load_sae(
    layer: int
):
    local_dir = "./jbloom_dictionaries"
    filename =  f"final_sparse_autoencoder_gpt2-small_blocks.{layer}.hook_resid_pre_24576.pt"

    save_path = f"{local_dir}/{filename}"
    sae = SparseAutoencoder.load_from_pretrained(save_path)
    sae.to("cuda:0")

    return sae

In [4]:
from working import ActivationCache
from nnsight import LanguageModel
from working.config import CacheConfig

layer = 8
feature_id = 6236 

model = LanguageModel("openai-community/gpt2", device_map="auto", dispatch=True)
sae = load_sae(
    layer=layer
)
cfg = CacheConfig

cache = ActivationCache(
    layer=layer,
    model=model,
    sae=sae,
    cfg = cfg
)

examples_list, max_act = cache.get_top_examples(feature_id)

 53%|█████▎    | 8/15 [00:19<00:19,  2.77s/it]

In [ ]:
n_debaters = 2

debaters = [
    Debater(
        get_client("openai", openai_key),
        f"debater {id}"
    ) 
    for id in range(n_debaters)
]

debate = CollabDebate(debaters, examples_list)

In [ ]:
debate.run(max_rounds=2)